In [0]:
bronze_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("includeExistingFiles", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option(
            "cloudFiles.schemaLocation",
            "/Volumes/nyc-taxi/default/nyc-taxi-data/_schema/bronze/"
        )
        .load("/Volumes/nyc-taxi/default/nyc-taxi-data/raw_data/")
)


In [0]:
from pyspark.sql.functions import current_timestamp, col

bronze_df = (
    bronze_df
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
bronze_stream = (
    bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/nyc-taxi/default/nyc-taxi-data/_checkpoints/bronze/")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable("`nyc-taxi`.bronze.raw_data")
)